<a href="https://colab.research.google.com/github/PauloCunhaJunior/modelogeradoapi/blob/main/modelogerado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import cifar10

# Carregar os dados CIFAR-10
(train_images, train_labels), (test_images, test_labels) = cifar10.load_data()

# Normalizar as imagens
train_images = train_images / 255.0
test_images = test_images / 255.0

# Criar o modelo CNN
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 3)),

    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

# Compilar o modelo
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Treinar o modelo
history = model.fit(
    train_images,
    train_labels,
    epochs=10,
    validation_data=(test_images, test_labels)
)

# Avaliar o modelo
test_loss, test_acc = model.evaluate(test_images, test_labels)

print("Acurácia de teste:", test_acc)

# Salvar o modelo treinado
model.save("modelo_cifar10.h5")

print("Modelo salvo com sucesso!")

Epoch 1/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 83s 51ms/step - accuracy: 0.4634 - loss: 1.4693 - val_accuracy: 0.5857 - val_loss: 1.1595
Epoch 2/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 76s 49ms/step - accuracy: 0.6118 - loss: 1.0962 - val_accuracy: 0.6451 - val_loss: 1.0056
Epoch 3/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 84s 50ms/step - accuracy: 0.6752 - loss: 0.9176 - val_accuracy: 0.6703 - val_loss: 0.9420
Epoch 4/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 83s 51ms/step - accuracy: 0.7160 - loss: 0.8054 - val_accuracy: 0.7053 - val_loss: 0.8496
Epoch 5/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 80s 51ms/step - accuracy: 0.7463 - loss: 0.7168 - val_accuracy: 0.6878 - val_loss: 0.8895
Epoch 6/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 77s 50ms/step - accuracy: 0.7733 - loss: 0.6425 - val_accuracy: 0.7175 - val_loss: 0.8428
Epoch 7/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 78s 50ms/step - accuracy: 0.7960 - loss: 0.5785 - val_accuracy: 0.7272 - val_loss: 0.8335
Epoch 8/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 85s 52ms/step - accuracy: 0.8189 -

Acurácia de teste: 0.7182999849319458
Modelo salvo com sucesso!


In [2]:
from flask import Flask, request, jsonify
import tensorflow as tf
import numpy as np
from PIL import Image
import io
import threading

app = Flask(__name__)

model = tf.keras.models.load_model("modelo_cifar10.h5")

class_names = [
    "avião",
    "automóvel",
    "pássaro",
    "gato",
    "cervo",
    "cachorro",
    "sapo",
    "cavalo",
    "navio",
    "caminhão"
]

@app.route("/", methods=["GET"])
def home():
    return """
    <h1>API CIFAR-10</h1>
    <p>API funcionando!</p>

    <h2>Enviar imagem para previsão</h2>

    <form action="/prever" method="post" enctype="multipart/form-data">
        <input type="file" name="imagem" accept="image/*">
        <br><br>
        <button type="submit">Enviar imagem</button>
    </form>
    """

@app.route("/prever", methods=["POST"])
def prever():
    try:
        if "imagem" not in request.files:
            return jsonify({
                "erro": "Nenhuma imagem foi enviada. Envie uma imagem no campo chamado 'imagem'."
            }), 400

        arquivo = request.files["imagem"]

        img = Image.open(io.BytesIO(arquivo.read()))
        img = img.convert("RGB")
        img = img.resize((32, 32))

        img_array = np.array(img)
        img_array = img_array / 255.0
        img_array = np.expand_dims(img_array, axis=0)

        prediction = model.predict(img_array)

        predicted_class = int(np.argmax(prediction))
        predicted_label = class_names[predicted_class]
        confidence = float(np.max(prediction))

        return jsonify({
            "classe_numero": predicted_class,
            "classe_nome": predicted_label,
            "confianca": round(confidence, 4)
        })

    except Exception as e:
        return jsonify({
            "erro": str(e)
        }), 500

# Rodar a API em segundo plano
def rodar_api():
    app.run(host="0.0.0.0", port=5000)

thread = threading.Thread(target=rodar_api)
thread.start()

In [7]:
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(5000)")
print("Acesse sua API por este link:")
print(url)

Acesse sua API por este link:
https://5000-m-s-kkb-ase1a0-2zzsnxylo1bea-a.asia-east1-0.prod.colab.dev


In [6]:
import os
import requests
import json

url_api = "http://127.0.0.1:5000/prever"
pasta_imagens = "/content/imagens"

print("Testando a API neste endereço:")
print(url_api)
print("-" * 50)

for arquivo in os.listdir(pasta_imagens):

    if arquivo.lower().endswith((".png", ".jpg", ".jpeg")):

        caminho_imagem = os.path.join(pasta_imagens, arquivo)

        print(f"Imagem enviada: {arquivo}")

        with open(caminho_imagem, "rb") as img:
            resposta = requests.post(
                url_api,
                files={"imagem": img}
            )

        if resposta.status_code == 200:
            dados = resposta.json()

            print(json.dumps(dados, indent=2, ensure_ascii=False))

        else:
            print("Erro ao enviar imagem.")
            print("Status:", resposta.status_code)
            print("Texto retornado:")
            print(resposta.text)

        print("-" * 50)

Testando a API neste endereço:
http://127.0.0.1:5000/prever
--------------------------------------------------
Imagem enviada: automóvel1.jpeg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step


INFO:werkzeug:127.0.0.1 - - [24/May/2026 15:37:09] "POST /prever HTTP/1.1" 200 -


{
  "classe_nome": "caminhão",
  "classe_numero": 9,
  "confianca": 0.9806
}
--------------------------------------------------
Imagem enviada: avião2.jpeg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step


INFO:werkzeug:127.0.0.1 - - [24/May/2026 15:37:09] "POST /prever HTTP/1.1" 200 -


{
  "classe_nome": "avião",
  "classe_numero": 0,
  "confianca": 0.9995
}
--------------------------------------------------
Imagem enviada: sapo2.jpeg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step


...

{
  "classe_nome": "cachorro",
  "classe_numero": 5,
  "confianca": 0.5512
}
--------------------------------------------------
Imagem enviada: cervo2.jpeg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step


INFO:werkzeug:127.0.0.1 - - [24/May/2026 15:37:11] "POST /prever HTTP/1.1" 200 -


{
  "classe_nome": "cervo",
  "classe_numero": 4,
  "confianca": 0.9996
}
--------------------------------------------------
Imagem enviada: navio2.jpeg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step


INFO:werkzeug:127.0.0.1 - - [24/May/2026 15:37:11] "POST /prever HTTP/1.1" 200 -


{
  "classe_nome": "navio",
  "classe_numero": 8,
  "confianca": 1.0
}
--------------------------------------------------
